In [1]:
# ============================================================================
# BLOCK 1: DATA LOADING - POLISH
# Input: pl_comments.csv
# Output: 01_loaded_data_pl.pkl
# ============================================================================

# CELL 1: Load configuration from setup notebook
# (Run 00_setup_and_config.ipynb first)
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 04:00:51
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
# CELL 2: Load raw data
print('='*70)
print('BLOCK 1: DATA LOADING - POLISH')
print('='*70)

# Check if checkpoint exists
if check_checkpoint_exists('pl', '01_loaded_data'):
    df_pl = load_checkpoint('pl', '01_loaded_data')
else:
    # Load raw CSV
    raw_file = RAW_DIR / 'polish_comments.csv'
    
    if not raw_file.exists():
        raise FileNotFoundError(f'Polish comments file not found: {raw_file}')
    
    print(f'\nLoading: {raw_file}')
    df_pl = pd.read_csv(raw_file)
    
    print(f'\n✓ Loaded {len(df_pl):,} rows × {len(df_pl.columns)} columns')
    
    # Standardize column names to snake_case
    def to_snake_case(col_name: str) -> str:
        col = str(col_name).strip().lower()
        col = re.sub(r'[^a-z0-9]+', '_', col)
        col = re.sub(r'_+', '_', col).strip('_')
        return col
    
    df_pl.columns = [to_snake_case(c) for c in df_pl.columns]
    
    print(f'\nColumns after standardization:')
    print(df_pl.columns.tolist())
    
    # Convert numeric columns
    for col in ['comment_likes', 'replies']:
        if col in df_pl.columns:
            df_pl[col] = pd.to_numeric(df_pl[col], errors='coerce')
    
    # Convert datetime
    if 'comment_date' in df_pl.columns:
        df_pl['comment_date'] = pd.to_datetime(df_pl['comment_date'], errors='coerce', utc=True)
    
    # Drop is_creator column (not needed)
    if 'is_creator' in df_pl.columns:
        df_pl = df_pl.drop(columns=['is_creator'])
        print('\n✓ Dropped is_creator column (not needed for analysis)')
    
    # Save checkpoint
    save_checkpoint(df_pl, 'pl', '01_loaded_data')
    update_pipeline_status('pl', 1, 'completed')

BLOCK 1: DATA LOADING - POLISH

Loading: data\raw\polish_comments.csv

✓ Loaded 6,891 rows × 7 columns

Columns after standardization:
['video_id', 'author_name', 'comment_text', 'comment_date', 'comment_likes', 'replies', 'is_creator']

✓ Dropped is_creator column (not needed for analysis)
✓ Checkpoint saved: 01_loaded_data_pl.pkl
✓ Pipeline status updated: pl - Block 1 - completed


In [3]:
# CELL 3: Data audit
print('\n' + '='*70)
print('DATA AUDIT - POLISH')
print('='*70)

audit_summary = {
    'Total Rows': len(df_pl),
    'Total Columns': len(df_pl.columns),
    'Exact Duplicates': int(df_pl.duplicated().sum()),
    'Total Missing Cells': int(df_pl.isna().sum().sum()),
    'Rows with Any Missing': int(df_pl.isna().any(axis=1).sum())
}

if 'video_id' in df_pl.columns:
    audit_summary['Unique Videos'] = int(df_pl['video_id'].nunique(dropna=True))
if 'author_name' in df_pl.columns:
    audit_summary['Unique Authors'] = int(df_pl['author_name'].nunique(dropna=True))

for key, value in audit_summary.items():
    print(f'  {key}: {value:,}' if isinstance(value, int) else f'  {key}: {value}')



DATA AUDIT - POLISH
  Total Rows: 6,891
  Total Columns: 6
  Exact Duplicates: 2
  Total Missing Cells: 5
  Rows with Any Missing: 5
  Unique Videos: 12
  Unique Authors: 6,138


In [4]:
# CELL 4: Preview
print('\nDataset Preview (first 5 rows):')
display(df_pl.head(5))

print('\n' + '='*70)
print('✓ BLOCK 1 COMPLETE - POLISH DATA LOADED')
print('='*70)


Dataset Preview (first 5 rows):


,video_id,author_name,comment_text,comment_date,comment_likes,replies
0,-rZ7h41aRNU,@andrzejs.6815,Niewiarygodne że człowiek który całe życie siedział przy piłce nożnej tłumaczy ludziom polityczn...,2025-06-03 08:56:53+00:00,15397,235
1,-rZ7h41aRNU,@bagins89,"Według mnie najgorsza jest pogarda lewej strony która po porażce nazywa połowę Polski betonem, i...",2025-06-03 09:37:43+00:00,6136,201
2,-rZ7h41aRNU,@djznykudjznyku9140,"Hipokryzja , zmienność poglądów , pogarda i pycha to są przyczyny porażki",2025-06-03 08:39:25+00:00,6040,58
3,-rZ7h41aRNU,@joku4303,"Mam w otoczeniu profesorów, doktorów, lekarzy, którzy głosowali na Nawrockiego, ale część z nich...",2025-06-03 09:15:36+00:00,3676,151
4,-rZ7h41aRNU,@magorzata8608,Panie Krzysztofie proszę zrobić komentarz o hejcie na małą Kasię Nawrocką bo to jak uśmiechnięci...,2025-06-03 08:52:41+00:00,3505,94



✓ BLOCK 1 COMPLETE - POLISH DATA LOADED
